# DeepFake Shield — hybrid temporal model (Colab T4)

EfficientNet-B0 frame encoder (frozen, the served checkpoint) + temporal Transformer trained on
precomputed embeddings. All logic lives in `scripts/extract_embeddings.py` and
`scripts/train_temporal_embeddings.py`; these cells only set paths and call them.

**Leakage control:** early stopping uses an inner-val set carved from *train*; the original *val*
split only tunes the decision threshold; *test* is touched once, for the final report.

Run top to bottom. Every step resumes from Drive if the runtime disconnects.

## a) Runtime, Drive, bundle, packages

In [ ]:
import subprocess, torch
print("PyTorch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.total", "--format=csv"], capture_output=True, text=True).stdout)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/deepfake_shield")    # colab_bundle.zip was uploaded here
BUNDLE_ZIP = DRIVE_ROOT / "colab_bundle.zip"         # first part; holds the manifest
WORK = Path("/content/work")                                    # fast local disk
EMBEDDINGS = DRIVE_ROOT / "embeddings_f16"                      # survives disconnects
EVALUATIONS = DRIVE_ROOT / "artifacts/evaluations"
EXPERIMENT = DRIVE_ROOT / "artifacts/experiments/temporal_transformer_f16"

# Built with --no-videos? Point this at the dataset copy already on Drive instead. Reading from
# the Drive mount is slow, so copying it to local disk first (next cell) is strongly recommended.
EXISTING_DRIVE_DATASET = Path("/content/drive/MyDrive/DeepFake-Shield/datasets/raw")
VIDEO_ROOT = WORK / "datasets/raw"

In [ ]:
import json, shutil, zipfile

# The bundle is split into colab_bundle.zip (code, splits, manifest, first videos) plus
# colab_bundle_part2.zip, ... (FAT32 caps files at 4 GiB). Upload every part to DRIVE_ROOT.
if not (WORK / ".bundle_extracted").is_file():
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(BUNDLE_ZIP) as first:
        expected = json.loads(first.read("BUNDLE_MANIFEST.json"))["parts"]
    missing = [name for name in expected if not (DRIVE_ROOT / name).is_file()]
    assert not missing, f"Upload these bundle parts to {DRIVE_ROOT}: {missing}"
    for name in expected:
        with zipfile.ZipFile(DRIVE_ROOT / name) as bundle:
            members = [m for m in bundle.namelist() if not Path(m).name.startswith("._")]
            print(f"Extracting {len(members)} files from {name} ...")
            bundle.extractall(WORK, members)
    (WORK / ".bundle_extracted").touch()
manifest = json.loads((WORK / "BUNDLE_MANIFEST.json").read_text())
print({key: manifest[key] for key in ("created_at", "git_commit", "videos_included", "videos", "split_sizes", "parts")})

if not manifest["videos_included"] and not VIDEO_ROOT.is_dir():
    print("Copying the Drive dataset to local disk (one-off, ~9 GB) ...")
    shutil.copytree(EXISTING_DRIVE_DATASET, VIDEO_ROOT, ignore=shutil.ignore_patterns("._*", ".DS_Store"))
print("videos on disk:", sum(1 for _ in VIDEO_ROOT.rglob("*.mp4")))


Colab ships OpenCV 5.x, which has **no `cv2.CascadeClassifier`** (the Haar face detector the model was
evaluated with). Replace it with the exact 4.14.0.94 build used for `baseline_f16`, so face boxes match.
If OpenCV was already imported in this runtime, restart the session after this cell.

In [ ]:
%pip uninstall -y -q opencv-python opencv-contrib-python opencv-contrib-python-headless opencv-python-headless
%pip install -q opencv-python-headless==4.14.0.94 scikit-learn matplotlib
import cv2
print("OpenCV", cv2.__version__, "| CascadeClassifier:", hasattr(cv2, "CascadeClassifier"))
assert hasattr(cv2, "CascadeClassifier"), "Restart the session (Runtime -> Restart session) and run from the top."

In [ ]:
import os, sys
os.chdir(WORK)
for path in (WORK / "scripts", WORK / "backend", WORK / "ml"):
    sys.path.insert(0, str(path))
import extract_embeddings as ee
import train_temporal_embeddings as tt

## b) Embedding extraction

Sanity check first: 20 val videos re-scored here must match `baseline_f16`'s cached predictions to
within 1e-3 (same face-frame count, same mean fake probability). If it fails, the cell stops; try
`--no-autocast` once to rule out fp16, and otherwise report the printed differences.

In [ ]:
COMMON = ["--out-dir", str(EMBEDDINGS), "--video-root", str(VIDEO_ROOT), "--frames", "16",
          "--workers", "2", "--videos-per-batch", "8"]
ee.main(COMMON + ["--sanity-only"])

In [ ]:
# ~6,000 videos x 16 frames. CPU decoding + Haar is the bottleneck on Colab's 2 vCPUs; shards of 500
# videos are written to Drive as they finish, so rerunning this cell resumes.
ee.main(COMMON)

In [ ]:
import numpy as np
for split in ("train", "val", "test"):
    with np.load(EMBEDDINGS / f"{split}.npz") as data:
        valid = data["mask"].sum(1)
        print(f"{split:5} videos {len(valid):5}  no-face {(valid == 0).sum():3}  mean face frames {valid.mean():.2f}  "
              f"real {(data['labels'] == 0).sum():4}  fake {(data['labels'] == 1).sum():4}")

## c) Aggregation ablation (no training)

Mean / max / top-3 mean / median of the saved per-frame baseline probabilities, threshold tuned on
val (Youden's J), reported on test. `baseline_f16_mean` must reproduce `baseline_f16`.

In [ ]:
results = tt.aggregation_ablation(EMBEDDINGS, EVALUATIONS, WORK / "artifacts/splits")
reference = __import__("json").loads((WORK / "artifacts/evaluations/baseline_f16/metrics.json").read_text())
print("baseline_f16 test ROC-AUC", reference["test"]["roc_auc"], "| mean aggregation", results["mean"]["test"]["roc_auc"])

## d–e) Temporal Transformer: train, early-stop on inner-val, evaluate

Projection 1280→256, learnable positions, 2 pre-norm encoder layers, 4 heads, dropout 0.2, masked
mean pooling, 2-class head. Class-weighted cross-entropy, AdamW (lr 1e-4, wd 1e-2), cosine schedule,
batch 32, seed 42, fp16 autocast. Early stopping: inner-val ROC-AUC, patience 10, max 100 epochs.

In [ ]:
temporal = tt.train_temporal(EMBEDDINGS, EXPERIMENT, EVALUATIONS / "temporal_f16", WORK / "artifacts/splits")
print("best epoch", temporal["best_epoch"], "of", temporal["stopped_epoch"])

In [ ]:
from IPython.display import Image, display
display(Image(str(EXPERIMENT / "training_curves.png")))
display(Image(str(EVALUATIONS / "temporal_f16/roc_curve.png")))

In [ ]:
import subprocess
shutil.copytree(WORK / "artifacts/evaluations/baseline_f16", EVALUATIONS / "baseline_f16", dirs_exist_ok=True)
print(subprocess.run([sys.executable, str(WORK / "scripts/make_ablation_table.py"), "--evaluations-dir", str(EVALUATIONS)],
                     capture_output=True, text=True).stdout)

## f) results_bundle.zip

Experiment folder + every new evaluation folder, with repo-relative paths (`artifacts/...`), so it
unzips straight into the repository. Embeddings are left out (large; they stay on Drive).

In [ ]:
RESULTS = DRIVE_ROOT / "results_bundle.zip"
folders = [EXPERIMENT] + [EVALUATIONS / name for name in
           ("baseline_f16_mean", "baseline_f16_max", "baseline_f16_top3", "baseline_f16_median", "temporal_f16")]
with zipfile.ZipFile(RESULTS, "w", zipfile.ZIP_DEFLATED) as bundle:
    for folder in folders:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and not path.name.startswith("._") and path.name != ".DS_Store":
                bundle.write(path, path.relative_to(DRIVE_ROOT))
    bundle.write(EMBEDDINGS / "meta.json", "artifacts/experiments/temporal_transformer_f16/embeddings_meta.json")
print(RESULTS, f"{RESULTS.stat().st_size / 1024**2:.1f} MiB")